# Introduction to Statistical Arbitrage & Pairs Trading

Pairs trading is a market-neutral strategy that models two historically linked assets. When their relative pricing diverges beyond statistical norms, we trade the expectation that their spread will revert to its historical mean.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
import yfinance as yf

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

### Step 1: Download Market Data

In [ ]:
tickers = ['CVX', 'XOM']
data = yf.download(tickers, start='2021-01-01', end='2024-01-01', auto_adjust=True)['Close']
data = data.dropna()

fig, ax = plt.subplots(figsize=(14, 5))
data['CVX'].plot(ax=ax, label='CVX', color='#1f77b4', lw=1.5)
data['XOM'].plot(ax=ax, label='XOM', color='#ff7f0e', lw=1.5)
ax.set_title('Asset Price Series: CVX vs XOM', fontsize=14, fontweight='bold')
ax.set_ylabel('Adjusted Close Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

### Step 2: Linear Regression & Spread Transformation

$$Y_t = \alpha + \beta X_t + \epsilon_t$$
$$\text{Spread}_t = Y_t - (\alpha + \beta X_t)$$

In [ ]:
X = sm.add_constant(data['XOM'])
y = data['CVX']
model = sm.OLS(y, X).fit()

alpha = model.params['const']
beta = model.params['XOM']
spread = y - (alpha + beta * data['XOM'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(data.index, y, label='CVX (Actual)', color='#1f77b4', lw=1.5)
ax1.plot(data.index, alpha + beta * data['XOM'], label=f'Model ({alpha:.2f} + {beta:.2f} * XOM)', color='#2ca02c', ls='--', lw=1.5)
ax1.set_title(f'OLS Linear Regression Fit (Hedge Ratio β = {beta:.4f})', fontsize=13, fontweight='bold')
ax1.set_ylabel('Price ($)')
ax1.legend()

ax2.plot(spread.index, spread, label='Spread Residual', color='#9467bd', lw=1.5)
ax2.axhline(spread.mean(), color='black', linestyle='--', label=f'Mean ({spread.mean():.2f})')
ax2.axhline(spread.mean() + spread.std(), color='red', linestyle=':', label='±1 Std Dev')
ax2.axhline(spread.mean() - spread.std(), color='red', linestyle=':')
ax2.axhline(spread.mean() + 2 * spread.std(), color='darkred', linestyle=':', lw=1.5, label='±2 Std Dev')
ax2.axhline(spread.mean() - 2 * spread.std(), color='darkred', linestyle=':', lw=1.5)
ax2.set_title('Transformed Residual Spread (Mean-Reverting Series)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Spread Value')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

### Step 3: Augmented Dickey-Fuller (ADF) Test

Testing for stationarity on the spread residual series.

In [ ]:
adf_result = adfuller(spread)

print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value:       {adf_result[1]:.4e}")
print("Critical Values:")
for key, val in adf_result[4].items():
    print(f"   {key}: {val:.4f}")

z_score = (spread - spread.mean()) / spread.std()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(z_score.index, z_score, color='#17becf', lw=1.2, label='Z-Score')
ax.axhline(0, color='black', linestyle='--', lw=1)
ax.axhline(1.0, color='orange', linestyle=':', label='±1.0σ Threshold')
ax.axhline(-1.0, color='orange', linestyle=':')
ax.axhline(2.0, color='red', linestyle='--', label='±2.0σ Signal Threshold')
ax.axhline(-2.0, color='red', linestyle='--')
ax.set_title('Spread Normalized Z-Score Series', fontsize=13, fontweight='bold')
ax.set_ylabel('Z-Score')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Step 4: Large Universe Screening - Pairwise Correlation & Cointegration

In [ ]:
universe = [
    'AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'AMD',
    'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS',
    'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'MPC', 'VLO', 'OXY',
    'KO', 'PEP', 'PG', 'WMT', 'COST',
    'V', 'MA', 'HD', 'LOW'
]
universe_data = yf.download(universe, start='2021-01-01', end='2024-01-01', auto_adjust=True)['Close'].dropna()

corr_matrix = universe_data.corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Pearson Correlation'}, linewidths=0.5)
ax.set_title(f'Universe Pairwise Correlation Matrix ({len(universe)} Assets)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
pairs_results = []
n = len(universe)
p_matrix = pd.DataFrame(np.ones((n, n)), index=universe, columns=universe)

for i in range(n):
    for j in range(i + 1, n):
        t1, t2 = universe[i], universe[j]
        score, pvalue, _ = coint(universe_data[t1], universe_data[t2])
        p_matrix.loc[t1, t2] = pvalue
        p_matrix.loc[t2, t1] = pvalue
        
        reg = sm.OLS(universe_data[t1], sm.add_constant(universe_data[t2])).fit()
        pairs_results.append({
            'Pair': f"{t1} / {t2}",
            'Stock 1': t1,
            'Stock 2': t2,
            'p-value': pvalue,
            'Correlation': corr_matrix.loc[t1, t2],
            'Hedge Ratio': reg.params[t2]
        })

pairs_df = pd.DataFrame(pairs_results).sort_values('p-value').reset_index(drop=True)
print(f"Screened {len(pairs_df)} Unique Pairs across {len(universe)} Assets.")
print(f"Pairs with p < 0.05: {(pairs_df['p-value'] < 0.05).sum()}")
print("\nTop 15 Most Cointegrated Pairs:")
print(pairs_df.head(15).to_string(index=False, formatters={'p-value': '{:.4f}'.format, 'Correlation': '{:.2f}'.format, 'Hedge Ratio': '{:.4f}'.format}))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [1.2, 1]})

mask = np.triu(np.ones_like(p_matrix, dtype=bool))
sns.heatmap(p_matrix, cmap='YlGn_r', vmin=0, vmax=0.1, mask=mask, ax=ax1, cbar_kws={'label': 'p-value (< 0.05 Cointegrated)'}, linewidths=0.2)
ax1.set_title(f'Engle-Granger Cointegration Matrix ({len(universe)} Assets)', fontsize=13, fontweight='bold')

top_pairs_display = pairs_df.head(15).copy()
colors = ['#2ca02c' if p < 0.05 else '#7f7f7f' for p in top_pairs_display['p-value']]
ax2.barh(top_pairs_display['Pair'][::-1], top_pairs_display['p-value'][::-1], color=colors[::-1])
ax2.axvline(0.05, color='red', linestyle='--', label='α = 0.05 Threshold')
ax2.set_title('Top 15 Cointegrated Pairs by p-value', fontsize=13, fontweight='bold')
ax2.set_xlabel('Cointegration p-value')
ax2.legend()

plt.tight_layout()
plt.show()

### Step 5: Visualizing the Top Cointegrated Pair

In [ ]:
top_pair = pairs_df.iloc[0]
s1, s2 = top_pair['Stock 1'], top_pair['Stock 2']

X_top = sm.add_constant(universe_data[s2])
y_top = universe_data[s1]
ols_top = sm.OLS(y_top, X_top).fit()

spread_top = y_top - (ols_top.params['const'] + ols_top.params[s2] * universe_data[s2])
z_top = (spread_top - spread_top.mean()) / spread_top.std()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(universe_data.index, spread_top, color='#2ca02c', lw=1.3, label=f'Spread ({s1} - β*{s2})')
ax1.axhline(spread_top.mean(), color='black', linestyle='--', lw=1, label='Mean')
ax1.set_title(f'Top Cointegrated Pair Spread: {s1} vs {s2} (p = {top_pair["p-value"]:.4f})', fontsize=13, fontweight='bold')
ax1.set_ylabel('Spread Value')
ax1.legend()

ax2.plot(z_top.index, z_top, color='#1f77b4', lw=1.2, label='Z-Score')
ax2.axhline(0, color='black', linestyle='--', lw=1)
ax2.axhline(2.0, color='red', linestyle='--', label='Upper Entry (+2σ)')
ax2.axhline(-2.0, color='red', linestyle='--', label='Lower Entry (-2σ)')
ax2.axhline(0.5, color='green', linestyle=':', label='Exit Band (±0.5σ)')
ax2.axhline(-0.5, color='green', linestyle=':')
ax2.set_title('Z-Score & Trading Thresholds', fontsize=13, fontweight='bold')
ax2.set_ylabel('Z-Score')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

### Bonus: Autoencoder Latent Compression (TensorFlow & Keras) & t-SNE Clustering

Using a TensorFlow/Keras Autoencoder to compress high-dimensional return histories into a low-dimensional latent space, then projecting with t-SNE to reveal structural peer clusters in large universes.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models

    returns = universe_data.pct_change().dropna()
    returns_norm = (returns - returns.mean()) / returns.std()
    X_train = returns_norm.T.values

    input_dim = X_train.shape[1]
    latent_dim = 8

    input_layer = layers.Input(shape=(input_dim,))
    encoded = layers.Dense(64, activation='relu')(input_layer)
    encoded = layers.Dense(32, activation='relu')(encoded)
    latent = layers.Dense(latent_dim, activation='linear', name='latent_space')(encoded)

    decoded = layers.Dense(32, activation='relu')(latent)
    decoded = layers.Dense(64, activation='relu')(decoded)
    output_layer = layers.Dense(input_dim, activation='linear')(decoded)

    autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
    encoder = models.Model(inputs=input_layer, outputs=latent)

    autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse')
    history = autoencoder.fit(X_train, X_train, epochs=250, batch_size=len(X_train), verbose=0)

    latent_space = encoder.predict(X_train, verbose=0)

    print(f"Compressed {len(universe)} stocks from {input_dim} features to {latent_dim}-D latent space.")
    print(f"TensorFlow / Keras Autoencoder MSE Loss: {history.history['loss'][-1]:.4f}")

except ImportError:
    from sklearn.decomposition import PCA
    returns = universe_data.pct_change().dropna()
    returns_norm = (returns - returns.mean()) / returns.std()
    latent_space = PCA(n_components=8, random_state=42).fit_transform(returns_norm.T.values)
    print("TensorFlow not found; compressed via 8-D linear bottleneck (PCA).")

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=5, random_state=42)
coords_2d = tsne.fit_transform(latent_space)

sector_map = {
    'AAPL': 'Tech', 'MSFT': 'Tech', 'GOOGL': 'Tech', 'META': 'Tech', 'NVDA': 'Tech', 'AMD': 'Tech',
    'JPM': 'Financials', 'BAC': 'Financials', 'WFC': 'Financials', 'C': 'Financials', 'GS': 'Financials', 'MS': 'Financials',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'SLB': 'Energy', 'EOG': 'Energy', 'MPC': 'Energy', 'VLO': 'Energy', 'OXY': 'Energy',
    'KO': 'Consumer/Retail', 'PEP': 'Consumer/Retail', 'PG': 'Consumer/Retail', 'WMT': 'Consumer/Retail', 'COST': 'Consumer/Retail', 'HD': 'Consumer/Retail', 'LOW': 'Consumer/Retail',
    'V': 'Payments', 'MA': 'Payments'
}

sectors = [sector_map.get(ticker, 'Other') for ticker in universe]
palette = {'Tech': '#1f77b4', 'Financials': '#ff7f0e', 'Energy': '#2ca02c', 'Consumer/Retail': '#d62728', 'Payments': '#9467bd'}

fig, ax = plt.subplots(figsize=(13, 9))
sns.scatterplot(x=coords_2d[:, 0], y=coords_2d[:, 1], hue=sectors, palette=palette, s=180, alpha=0.9, ax=ax)

for i, ticker in enumerate(universe):
    ax.annotate(ticker, (coords_2d[i, 0] + 0.3, coords_2d[i, 1] + 0.3), fontsize=11, fontweight='bold', alpha=0.85)

ax.set_title('Keras/TensorFlow Autoencoder Latent Space Projected via t-SNE', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(title='Sector Cluster', loc='best')
plt.tight_layout()
plt.show()